# Forecast-driven dispatch

Daily schedules built from forecast prices only and settled at realised 2024 prices.


## Fair evaluation boundary

The following information is separated deliberately:

- **Forecast price:** used by the optimiser to choose charge and discharge quantities.
- **Realised price:** used only afterward to calculate the simulated cash result.
- **Initial and terminal SoC:** both fixed at 0.8 MWh for every day, preventing the strategy from borrowing value from the next day or consuming initial inventory.

This daily reset is a simplifying design choice. A later rolling-horizon version can carry state of charge between days, but it must preserve the same information boundary and audit checks.

In [1]:
from pathlib import Path

import cvxpy as cp
import numpy as np
import pandas as pd

FORECAST_DATA_PATH = Path("data/forecast_test_2024.csv")
DISPATCH_LOG_PATH = Path("data/dispatch_test_2024.csv")

forecast_data = pd.read_csv(FORECAST_DATA_PATH, parse_dates=["timestamp"])
forecast_data["timestamp"] = pd.to_datetime(
    forecast_data["timestamp"], utc=True, errors="raise"
)
forecast_data = forecast_data.sort_values("timestamp").reset_index(drop=True)

assert forecast_data["timestamp"].is_monotonic_increasing
assert not forecast_data["timestamp"].duplicated().any()
assert forecast_data.isna().sum().sum() == 0
forecast_data.head()


,timestamp,price_eur_per_mwh,forecast_yesterday_eur_per_mwh,forecast_model_eur_per_mwh
0,2024-01-01 00:00:00+00:00,0.01,8.33,10.381210
1,2024-01-01 01:00:00+00:00,0.00,4.00,5.930276
2,2024-01-01 02:00:00+00:00,-0.01,0.31,6.111606
3,2024-01-01 03:00:00+00:00,-0.03,-0.06,6.388514
4,2024-01-01 04:00:00+00:00,-0.02,-0.04,7.209996


## Battery assumptions

These are unchanged from the previous battery lessons so that performance comparisons are meaningful. `charge_mwh` and `discharge_mwh` are grid-side hourly energies; the state update applies one-way charge and discharge efficiencies.

In [2]:
CAPACITY_MWH = 2.0
POWER_LIMIT_MW = 1.0
MIN_SOC_MWH = 0.10 * CAPACITY_MWH
MAX_SOC_MWH = 0.90 * CAPACITY_MWH
CHARGE_EFFICIENCY = 0.92
DISCHARGE_EFFICIENCY = 0.92
WEAR_EUR_PER_MWH_DISCHARGED = 5.0
STEP_HOURS = 1.0
DAILY_INITIAL_SOC_MWH = 0.8
TOLERANCE = 1e-7


## Daily scheduling function


In [3]:
def schedule_one_day(day_frame: pd.DataFrame, forecast_column: str) -> pd.DataFrame:
    """Create one 24-hour feasible schedule from forecasts and settle it at actual prices."""
    if len(day_frame) != 24:
        raise ValueError(f"Expected 24 hourly rows, received {len(day_frame)}")

    forecast_price = day_frame[forecast_column].to_numpy(dtype=float)
    actual_price = day_frame["price_eur_per_mwh"].to_numpy(dtype=float)
    hours = len(day_frame)

    charge_mwh = cp.Variable(hours, nonneg=True)
    discharge_mwh = cp.Variable(hours, nonneg=True)
    soc_mwh = DAILY_INITIAL_SOC_MWH + cp.cumsum(
        CHARGE_EFFICIENCY * charge_mwh - discharge_mwh / DISCHARGE_EFFICIENCY
    )

    constraints = [
        charge_mwh <= POWER_LIMIT_MW * STEP_HOURS,
        discharge_mwh <= POWER_LIMIT_MW * STEP_HOURS,
        soc_mwh >= MIN_SOC_MWH,
        soc_mwh <= MAX_SOC_MWH,
        soc_mwh[-1] == DAILY_INITIAL_SOC_MWH,
    ]

    forecast_contribution = (
        forecast_price @ (discharge_mwh - charge_mwh)
        - WEAR_EUR_PER_MWH_DISCHARGED * cp.sum(discharge_mwh)
    )
    problem = cp.Problem(cp.Maximize(forecast_contribution), constraints)
    problem.solve(solver="CLARABEL")

    if problem.status not in {"optimal", "optimal_inaccurate"}:
        day = day_frame["timestamp"].iloc[0].date()
        raise RuntimeError(f"Solver failed for {day}: {problem.status}")

    result = day_frame[[
        "timestamp",
        "price_eur_per_mwh",
        "forecast_yesterday_eur_per_mwh",
        "forecast_model_eur_per_mwh",
    ]].copy()
    result["charge_mwh"] = charge_mwh.value
    result["discharge_mwh"] = discharge_mwh.value
    result["soc_mwh"] = soc_mwh.value
    result["forecast_contribution_eur"] = (
        forecast_price * (discharge_mwh.value - charge_mwh.value)
        - WEAR_EUR_PER_MWH_DISCHARGED * discharge_mwh.value
    )
    result["purchase_cost_eur"] = actual_price * charge_mwh.value
    result["sales_revenue_eur"] = actual_price * discharge_mwh.value
    result["wear_cost_eur"] = WEAR_EUR_PER_MWH_DISCHARGED * discharge_mwh.value
    result["realised_contribution_eur"] = (
        result["sales_revenue_eur"]
        - result["purchase_cost_eur"]
        - result["wear_cost_eur"]
    )
    result["solver_status"] = problem.status
    return result


## Simulate 2024 with the ML forecast


In [4]:
daily_ml_logs = []
for _, day_frame in forecast_data.groupby(forecast_data["timestamp"].dt.floor("D"), sort=True):
    daily_ml_logs.append(
        schedule_one_day(day_frame, forecast_column="forecast_model_eur_per_mwh")
    )

ml_dispatch_log = pd.concat(daily_ml_logs, ignore_index=True)
print(f"Simulated {ml_dispatch_log['timestamp'].dt.date.nunique()} daily schedules.")
ml_dispatch_log.head()


Simulated 366 daily schedules.


,timestamp,price_eur_per_mwh,forecast_yesterday_eur_per_mwh,forecast_model_eur_per_mwh,charge_mwh,discharge_mwh,soc_mwh,forecast_contribution_eur,purchase_cost_eur,sales_revenue_eur,wear_cost_eur,realised_contribution_eur,solver_status
0,2024-01-01 00:00:00+00:00,0.01,8.33,10.381210,1.884938e-13,5.520000e-01,0.2,2.970428e+00,1.884938e-15,5.520000e-03,2.760000e+00,-2.754480e+00,optimal
1,2024-01-01 01:00:00+00:00,0.00,4.00,5.930276,2.912744e-11,2.334417e-12,0.2,-1.705621e-10,0.000000e+00,0.000000e+00,1.167208e-11,-1.167208e-11,optimal
2,2024-01-01 02:00:00+00:00,-0.01,0.31,6.111606,1.924159e-11,2.763737e-12,0.2,-1.145248e-10,-1.924159e-13,-2.763737e-14,1.381868e-11,-1.365390e-11,optimal
3,2024-01-01 03:00:00+00:00,-0.03,-0.06,6.388514,1.415280e-11,3.395278e-12,0.2,-8.570099e-11,-4.245841e-13,-1.018583e-13,1.697639e-11,-1.665366e-11,optimal
4,2024-01-01 04:00:00+00:00,-0.02,-0.04,7.209996,7.321087e-12,6.009256e-12,0.2,-3.950458e-11,-1.464217e-13,-1.201851e-13,3.004628e-11,-3.002004e-11,optimal


## Validate every schedule


In [5]:
assert len(ml_dispatch_log) == len(forecast_data)
assert set(ml_dispatch_log["solver_status"]).issubset({"optimal", "optimal_inaccurate"})
assert (ml_dispatch_log["charge_mwh"] >= -TOLERANCE).all()
assert (ml_dispatch_log["discharge_mwh"] >= -TOLERANCE).all()
assert (ml_dispatch_log["charge_mwh"] <= POWER_LIMIT_MW * STEP_HOURS + TOLERANCE).all()
assert (ml_dispatch_log["discharge_mwh"] <= POWER_LIMIT_MW * STEP_HOURS + TOLERANCE).all()
assert (ml_dispatch_log["soc_mwh"] >= MIN_SOC_MWH - TOLERANCE).all()
assert (ml_dispatch_log["soc_mwh"] <= MAX_SOC_MWH + TOLERANCE).all()

terminal_soc_by_day = ml_dispatch_log.groupby(
    ml_dispatch_log["timestamp"].dt.floor("D")
)["soc_mwh"].last()
assert np.isclose(terminal_soc_by_day, DAILY_INITIAL_SOC_MWH, atol=TOLERANCE).all()

maximum_simultaneous_mwh = np.minimum(
    ml_dispatch_log["charge_mwh"], ml_dispatch_log["discharge_mwh"]
).max()
assert maximum_simultaneous_mwh < TOLERANCE

print("All dispatch validation checks passed.")
print(f"Largest terminal SoC error: {(terminal_soc_by_day - DAILY_INITIAL_SOC_MWH).abs().max():.2e} MWh")
print(f"Maximum simultaneous charge/discharge: {maximum_simultaneous_mwh:.2e} MWh")


All dispatch validation checks passed.
Largest terminal SoC error: 8.06e-10 MWh
Maximum simultaneous charge/discharge: 3.74e-09 MWh


## Compare forecast choices fairly


In [6]:
daily_yesterday_logs = []
for _, day_frame in forecast_data.groupby(forecast_data["timestamp"].dt.floor("D"), sort=True):
    daily_yesterday_logs.append(
        schedule_one_day(day_frame, forecast_column="forecast_yesterday_eur_per_mwh")
    )

yesterday_dispatch_log = pd.concat(daily_yesterday_logs, ignore_index=True)


def dispatch_summary(name: str, log: pd.DataFrame) -> dict:
    """Aggregate realised financial results for one fixed dispatch method."""
    return {
        "strategy": name,
        "purchase_cost_eur": log["purchase_cost_eur"].sum(),
        "sales_revenue_eur": log["sales_revenue_eur"].sum(),
        "wear_cost_eur": log["wear_cost_eur"].sum(),
        "realised_contribution_eur": log["realised_contribution_eur"].sum(),
        "charge_mwh": log["charge_mwh"].sum(),
        "discharge_mwh": log["discharge_mwh"].sum(),
    }

comparison = pd.DataFrame([
    {
        "strategy": "No action",
        "purchase_cost_eur": 0.0,
        "sales_revenue_eur": 0.0,
        "wear_cost_eur": 0.0,
        "realised_contribution_eur": 0.0,
        "charge_mwh": 0.0,
        "discharge_mwh": 0.0,
    },
    dispatch_summary("Same hour yesterday forecast", yesterday_dispatch_log),
    dispatch_summary("ML forecast", ml_dispatch_log),
])

financial_columns = [
    "purchase_cost_eur",
    "sales_revenue_eur",
    "wear_cost_eur",
    "realised_contribution_eur",
    "charge_mwh",
    "discharge_mwh",
]
comparison[financial_columns] = comparison[financial_columns].round(2)
comparison


,strategy,purchase_cost_eur,sales_revenue_eur,wear_cost_eur,realised_contribution_eur,charge_mwh,discharge_mwh
0,No action,0.00,0.00,0.00,0.00,0.00,0.00
1,Same hour yesterday forecast,54080.55,92294.75,4874.19,33340.01,1151.75,974.84
2,ML forecast,51742.92,89939.60,4445.54,33751.14,1050.46,889.11


## Preserve the audit trail


In [7]:
ml_dispatch_log.to_csv(DISPATCH_LOG_PATH, index=False)
print(f"Saved {len(ml_dispatch_log):,} hourly dispatch records to {DISPATCH_LOG_PATH}")

ml_dispatch_log.loc[
    (ml_dispatch_log["charge_mwh"] > TOLERANCE)
    | (ml_dispatch_log["discharge_mwh"] > TOLERANCE),
    [
        "timestamp",
        "price_eur_per_mwh",
        "forecast_model_eur_per_mwh",
        "charge_mwh",
        "discharge_mwh",
        "soc_mwh",
        "realised_contribution_eur",
    ],
].head(20).round(4)


Saved 8,784 hourly dispatch records to data/dispatch_test_2024.csv


,timestamp,price_eur_per_mwh,forecast_model_eur_per_mwh,charge_mwh,discharge_mwh,soc_mwh,realised_contribution_eur
0,2024-01-01 00:00:00+00:00,0.01,10.3812,0.0000,0.552,0.200,-2.7545
12,2024-01-01 12:00:00+00:00,2.00,-0.3282,1.0000,0.000,1.120,-2.0000
13,2024-01-01 13:00:00+00:00,1.35,3.2686,0.7391,0.000,1.800,-0.9978
17,2024-01-01 17:00:00+00:00,53.76,27.8279,0.0000,0.472,1.287,23.0147
18,2024-01-01 18:00:00+00:00,56.77,27.8417,0.0000,1.000,0.200,51.7700
22,2024-01-01 22:00:00+00:00,35.55,11.6098,0.6522,0.000,0.800,-23.1848
26,2024-01-02 02:00:00+00:00,7.35,9.5079,0.0870,0.000,0.880,-0.6391
27,2024-01-02 03:00:00+00:00,4.19,8.8062,1.0000,0.000,1.800,-4.1900
41,2024-01-02 17:00:00+00:00,66.62,65.2138,0.0000,0.472,1.287,29.0846
42,2024-01-02 18:00:00+00:00,64.36,66.9146,0.0000,1.000,0.200,59.3600
